# Poisson Regression with a Regularized Horseshoe Prior: Real Data (Communities & Crime)

Applies the beta-Bayes calibration framework (well-specified threshold tau*, observed
discrepancy d_obs, root-found beta*) to the same real dataset Bag_code's
`sparse_poisson_regression_overlap.py` used (`crime.csv`, from the UCI Communities &
Crime dataset, n=1994, p=100 covariates, violent-crime counts as the Poisson response).

**Model/prior note**: Bag_code's actual Stan model file (`sparse_poisson_regression.stan`)
is missing from disk, so its exact prior hyperparameters can't be recovered. This
notebook reuses our own already-validated regularized (Piironen-Vehtari) horseshoe
model from `poisson_horseshoe_utils.py` -- the same structure their `--model horseshoe`
choice implies, just implemented in NumPyro instead of Stan.

**p0 note**: the horseshoe prior's global-scale hyperparameter tau0 requires a prior
guess p0 at the number of truly nonzero coefficients. A cross-validated Lasso
pre-screen was attempted but didn't converge reliably at this scale; p0=10 (a
moderate ~10% sparsity assumption, consistent with prior published analyses of this
dataset) is used instead as a literature-informed default.

**NUTS settings**: Bag_code's own Stan config used `adapt_delta=0.99, max_treedepth=15`
(more conservative than NumPyro's NUTS defaults of `target_accept_prob=0.8,
max_tree_depth=10`). Verified empirically that this dataset needs the more
conservative settings -- one block went from 100% divergent transitions at NumPyro's
defaults to 0% with Bag_code's settings -- so `target_accept_prob=0.99,
max_tree_depth=15` is used throughout this notebook.

**Progress logging note**: this notebook's fits are slow enough (multiple hours total)
that live progress visibility matters. Every fitting function below writes progress
to `_real_data_progress.log` in addition to its normal cell output, so progress can
be tailed live from a terminal while this notebook executes -- nbclient (the tool
used to run this notebook end-to-end) only writes a notebook's own cell outputs to
disk once the ENTIRE notebook finishes, not incrementally per cell.

**Methodology**: matches Cal Housing / Appliances Energy exactly -- observed
discrepancy via a within-dataset K=8 block U-statistic (jackknife SE), and the
well-specified threshold via Definition 1 (Q = posterior): simulate synthetic data
from the model's own likelihood at parameters drawn from a reference posterior fit
on the full dataset.

In [1]:
import time
import numpy as np
from jax import random
from sklearn.preprocessing import scale

import poisson_horseshoe_utils as phu

LOG_FILE = '_real_data_progress.log'
open(LOG_FILE, 'w').close()  # reset the log at the start of a fresh run

TARGET_ACCEPT = 0.99   # matches Bag_code's Stan adapt_delta=0.99
MAX_TREE_DEPTH = 15    # matches Bag_code's Stan max_treedepth=15
NUM_WARMUP = 500
NUM_SAMPLES = 500
K_BLOCKS = 8
P0 = 10                # prior guess at number of nonzero coefficients (see note above)
R_TAU_STAR = 30        # Monte Carlo draws for the well-specified threshold

XY = np.genfromtxt('/Users/alya57/Desktop/Bag_code/crime.csv', delimiter=',')
y_full = XY[:, 0].astype(int)
X_full = scale(XY[:, 1:])
n_full, p = X_full.shape
phu._log(f'Full dataset: n={n_full}, p={p}', LOG_FILE)

blocks = phu.make_k_blocks(X_full, y_full, K=K_BLOCKS, seed=42)
block_size = len(blocks[0][1])
phu._log(f'K={K_BLOCKS} blocks, size~{block_size}', LOG_FILE)

tau0_block = phu.tau0_default(p, p0=P0, n=block_size)
tau0_full = phu.tau0_default(p, p0=P0, n=n_full)
model_std_block = phu.make_model(beta=1.0, tau0=tau0_block)
model_std_full = phu.make_model(beta=1.0, tau0=tau0_full)

# Canonical posterior-cache convention shared by every block-level fit at a
# given beta (d_obs_std, beta* search, Tables 4/5) -- same beta + same blocks
# always maps to the same file + same seed, so whichever caller runs first
# computes it once and everyone else reuses it.
POSTERIORS_SEED = 4000
def posteriors_path(bval):
    beta_val = 1.0 if bval is None else float(bval)
    return f'_posteriors_beta_{beta_val:.4f}.pkl'

Full dataset: n=1994, p=100


K=8 blocks, size~250


/Users/alya57/venvs/myenv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Observed discrepancy (d_obs_std): within-dataset block U-statistic

Fit standard Bayes (beta=1) independently on each of the K=8 real-data blocks,
then compute the bridge-BC discrepancy for all C(8,2)=28 block pairs. SE via
leave-one-block-out jackknife.

In [2]:
def _compute_d_obs_std():
    fits_std = phu.load_or_fit_blocks(posteriors_path(None), model_std_block, blocks,
                                       num_warmup=NUM_WARMUP, num_samples=NUM_SAMPLES, base_seed_mcmc=POSTERIORS_SEED,
                                       target_accept_prob=TARGET_ACCEPT, max_tree_depth=MAX_TREE_DEPTH,
                                       log_file=LOG_FILE, label='d_obs_std')
    result = phu.block_u_statistic(model_std_block, blocks, fits_std)
    return {'mean_d': result['mean_d'], 'se_jk': result['se_jk'], 'div_rate': result['div_rate']}

phu._log('\n=== STAGE: d_obs_std (observed discrepancy, standard Bayes) ===', LOG_FILE)
result_d_obs_std = phu.load_or_compute_json('_checkpoint_d_obs_std.json', _compute_d_obs_std, LOG_FILE, label='d_obs_std')
phu._log(f"d_obs_std = {result_d_obs_std['mean_d']:.4f} (jackknife SE={result_d_obs_std['se_jk']:.4f}), "
         f"divergence rate={result_d_obs_std['div_rate']:.4f}", LOG_FILE)


=== STAGE: d_obs_std (observed discrepancy, standard Bayes) ===


[d_obs_std] Loaded cached result from _checkpoint_d_obs_std.json (skipping computation).


d_obs_std = 511.7232 (jackknife SE=10.2577), divergence rate=0.0000


## Well-specified threshold (tau*): Q = posterior (Definition 1)

Fit one reference standard-Bayes posterior on the FULL dataset (n=1994). For
R_TAU_STAR Monte Carlo draws, sample theta_m=(b0,b) from that reference posterior,
simulate two independent synthetic Poisson datasets on one block's X (matching the
block size used above) at theta_m, fit standard Bayes on each synthetic replicate,
and compute their bridge-BC discrepancy. Averaged over R_TAU_STAR draws.

In [3]:
from numpyro.infer import MCMC, NUTS
# (The reference-posterior fit needed for tau*'s synthetic replicates is folded into
# tau*'s own checkpointed compute_fn below, so it's only ever run if tau*'s checkpoint
# is missing -- no need for a separate always-executed cell here.)

In [4]:
def _compute_tau_star():
    def _fit_reference():
        t0 = time.time()
        kernel_ref = NUTS(model_std_full, target_accept_prob=TARGET_ACCEPT, max_tree_depth=MAX_TREE_DEPTH)
        mcmc_ref = MCMC(kernel_ref, num_warmup=NUM_WARMUP, num_samples=NUM_SAMPLES, num_chains=1, progress_bar=False)
        mcmc_ref.run(random.PRNGKey(42), X=X_full, y=y_full)
        div_rate = float(np.asarray(mcmc_ref.get_extra_fields()['diverging']).mean())
        phu._log(f'  reference posterior fit done in {(time.time()-t0)/60:.1f}min. div_rate={div_rate:.4f}', LOG_FILE)
        return mcmc_ref.get_samples(), div_rate

    phu._log('  fitting reference posterior on full dataset (n=%d)...' % n_full, LOG_FILE)
    ref_samples, ref_div_rate = phu.load_or_compute_pickle(
        '_posteriors_reference_full.pkl', _fit_reference, LOG_FILE, label='reference')

    X_block_template = blocks[0][0]
    pairs_wellspec = phu.generate_wellspec_pairs_from_posterior(
        ref_samples, X_block_template, R=R_TAU_STAR, base_seed=200)
    result = phu.pairwise_discrepancy_serial(
        model_std_block, pairs_wellspec, num_warmup=NUM_WARMUP, num_samples=NUM_SAMPLES,
        base_seed_mcmc=2000, collect_meff=True, p_dim=p,
        target_accept_prob=TARGET_ACCEPT, max_tree_depth=MAX_TREE_DEPTH, log_file=LOG_FILE,
        checkpoint_prefix='_posteriors_tau_star')
    return {
        'mean_d': result['mean_d'], 'std_d': result['std_d'], 'se_d': result['se_d'],
        'div_rate': result['div_rate'], 'm_eff_mean': result['m_eff_mean'],
        'm_eff_se': result['m_eff_se'], 'D_eff': result['D_eff'],
        'tau_pred_from_meff': result['tau_pred_from_meff'],
    }

phu._log('\n=== STAGE: tau* (well-specified threshold, R=%d Monte Carlo draws) ===' % R_TAU_STAR, LOG_FILE)
result_tau_star = phu.load_or_compute_json('_checkpoint_tau_star.json', _compute_tau_star, LOG_FILE, label='tau_star')
phu._log(f"tau* = {result_tau_star['mean_d']:.4f} (SE={result_tau_star['se_d']:.4f}), "
         f"divergence rate={result_tau_star['div_rate']:.4f}", LOG_FILE)
phu._log(f"m_eff = {result_tau_star['m_eff_mean']:.3f} +/- {result_tau_star['m_eff_se']:.3f}, "
         f"D_eff = {result_tau_star['D_eff']:.3f}, tau* predicted = {result_tau_star['tau_pred_from_meff']:.3f}", LOG_FILE)


=== STAGE: tau* (well-specified threshold, R=30 Monte Carlo draws) ===


[tau_star] Loaded cached result from _checkpoint_tau_star.json (skipping computation).


tau* = 36.4090 (SE=1.3925), divergence rate=0.0000


m_eff = 67.876 +/- 0.266, D_eff = 68.876, tau* predicted = 34.438


## beta* root-find

Evaluate the observed beta-Bayes discrepancy (same K=8 block U-statistic
methodology, at a given beta) on the real data, and find beta* such that it
matches tau*, via a bracket + brentq search (same precision approach used
elsewhere in the thesis).

In [5]:
from scipy.optimize import brentq

def _compute_beta_star():
    target_val = result_tau_star['mean_d']
    _eval_count = [0]

    def eval_observed_beta_discrepancy(beta_val):
        _eval_count[0] += 1
        phu._log(f'--- beta* eval #{_eval_count[0]}: fitting all {K_BLOCKS} blocks at beta={beta_val:.4f} ---', LOG_FILE)
        model_beta = phu.make_model(beta=float(beta_val), tau0=tau0_block, k_trunc=250)
        fits_beta = phu.load_or_fit_blocks(posteriors_path(beta_val), model_beta, blocks,
                                        num_warmup=NUM_WARMUP, num_samples=NUM_SAMPLES, base_seed_mcmc=POSTERIORS_SEED,
                                        target_accept_prob=TARGET_ACCEPT, max_tree_depth=MAX_TREE_DEPTH,
                                        log_file=LOG_FILE, label=f'beta={beta_val:.4f}')
        result = phu.block_u_statistic(model_beta, blocks, fits_beta)
        return result['mean_d'], result['se_jk'], result['div_rate']

    def gap(beta_val):
        d, _, _ = eval_observed_beta_discrepancy(beta_val)
        g = d - target_val
        phu._log(f'  gap({beta_val:.4f}) = {g:+.4f}  (d_obs_beta={d:.4f}, target={target_val:.4f})', LOG_FILE)
        return g

    beta_lo, beta_hi = 1.0, 1.5
    gap_lo = gap(beta_lo)
    gap_hi = gap(beta_hi)

    if gap_lo * gap_hi >= 0:
        phu._log('No sign change on [1.0, 1.5]; widen the bracket.', LOG_FILE)
        return {'beta_star': None}

    beta_star_val = float(brentq(gap, beta_lo, beta_hi, xtol=1e-3, rtol=1e-3, maxiter=15))
    d_beta_star, se_beta_star, div_beta_star = eval_observed_beta_discrepancy(beta_star_val)
    return {
        'beta_star': beta_star_val, 'd_obs_beta': d_beta_star,
        'se_obs_beta': se_beta_star, 'div_rate': div_beta_star,
        'target_tau_star': target_val,
    }

phu._log('\n=== STAGE: beta* root-find ===', LOG_FILE)
beta_star_result = phu.load_or_compute_json('_checkpoint_beta_star.json', _compute_beta_star, LOG_FILE, label='beta_star')
beta_star = beta_star_result['beta_star']
if beta_star is not None:
    phu._log(f"\nbeta* = {beta_star:.4f}", LOG_FILE)
    phu._log(f"d_obs_beta(beta*) = {beta_star_result['d_obs_beta']:.4f} (SE={beta_star_result['se_obs_beta']:.4f}), "
             f"div_rate={beta_star_result['div_rate']:.4f}", LOG_FILE)
    phu._log(f"target (tau*) = {beta_star_result['target_tau_star']:.4f}", LOG_FILE)


=== STAGE: beta* root-find ===


[beta_star] Loaded cached result from _checkpoint_beta_star.json (skipping computation).



beta* = 1.1810


d_obs_beta(beta*) = 36.1353 (SE=2.8235), div_rate=0.0000


target (tau*) = 36.4090


## Summary

In [6]:
summary = []
summary.append('=' * 70)
summary.append(f"{'Measure':<30}{'mean d':>12}{'SE':>10}{'div rate':>12}")
summary.append('-' * 70)
summary.append(f"{'tau* (well-specified)':<30}{result_tau_star['mean_d']:>12.4f}{result_tau_star['se_d']:>10.4f}{result_tau_star['div_rate']:>12.4f}")
summary.append(f"{'tau* predicted (m_eff)':<30}{result_tau_star['tau_pred_from_meff']:>12.4f}{'':>10}{'':>12}")
summary.append(f"{'d_obs_std (beta=1)':<30}{result_d_obs_std['mean_d']:>12.4f}{result_d_obs_std['se_jk']:>10.4f}{result_d_obs_std['div_rate']:>12.4f}")
if beta_star is not None:
    summary.append(f"{'d_obs_beta(beta*)':<30}{beta_star_result['d_obs_beta']:>12.4f}{beta_star_result['se_obs_beta']:>10.4f}{beta_star_result['div_rate']:>12.4f}")
    summary.append(f"{'beta*':<30}{beta_star:>12.4f}")
summary.append('=' * 70)
for line in summary:
    phu._log(line, LOG_FILE)

Measure                             mean d        SE    div rate


----------------------------------------------------------------------


tau* (well-specified)              36.4090    1.3925      0.0000


tau* predicted (m_eff)             34.4382                      


d_obs_std (beta=1)                511.7232   10.2577      0.0000


d_obs_beta(beta*)                  36.1353    2.8235      0.0000


beta*                               1.1810


## Downstream stability/accuracy tables (matching Cal Housing's Tables 4-6)

Fit 5 models (Standard Bayes, beta=1.05, beta*, beta=1.3, beta=1.5) on the same
K=8 real-data blocks, then compute:

- **Table 4** (parameter-inference stability): mean L2 distance between posterior
  means of (b0,b) and mean Frobenius norm between posterior covariances, across
  all 28 block pairs.
- **Table 5** (predictive-inference stability): mean d_1/2 = -2 log BC between
  posterior-predictive MEAN FUNCTIONS mu(x)=exp(b0+x^T b) at a fixed pooled
  covariate grid, approximated as multivariate Gaussian (no closed-form joint
  predictive exists for Poisson counts, unlike the NIG/Gaussian case).
- **Table 6** (leave-one-block-out predictive accuracy): MLPD, RMSE, Cov90/95/99,
  CRPS, all via Monte Carlo using posterior-predictive count draws.

In [7]:
BETA_MODELS = [
    ('Standard Bayes', None),
    ('beta=1.05', 1.05),
    (f'beta*={beta_star:.4f}', beta_star),
    ('beta=1.3', 1.3),
    ('beta=1.5', 1.5),
]

def _compute_tables45():
    phu._log('\n=== STAGE: fitting all 5 models on the 8 real blocks (Tables 4-5) ===', LOG_FILE)
    all_fits = {}
    for label, bval in BETA_MODELS:
        model_this = model_std_block if bval is None else phu.make_model(beta=float(bval), tau0=tau0_block, k_trunc=250)
        fits_this = phu.load_or_fit_blocks(posteriors_path(bval), model_this, blocks,
                                                num_warmup=NUM_WARMUP, num_samples=NUM_SAMPLES, base_seed_mcmc=POSTERIORS_SEED,
                                                target_accept_prob=TARGET_ACCEPT, max_tree_depth=MAX_TREE_DEPTH,
                                                log_file=LOG_FILE, label=label)
        all_fits[label] = fits_this
        phu._log(f'  done fitting: {label}', LOG_FILE)

    phu._log('\n=== Table 4: parameter-inference stability ===', LOG_FILE)
    t4 = {}
    for label, bval in BETA_MODELS:
        t4[label] = phu.parameter_stability_table(all_fits[label])
        phu._log(f"  {label:<20s}  mean_L2={t4[label]['mean_l2']:.4f}  "
                 f"mean_Frobenius={t4[label]['mean_frobenius']:.4f}", LOG_FILE)

    phu._log('\n=== Table 5: predictive-inference stability ===', LOG_FILE)
    rng_eval = np.random.default_rng(42)
    X_pool = np.vstack([Xb for Xb, yb in blocks])
    X_star = X_pool[rng_eval.choice(len(X_pool), size=100, replace=False)]
    t5 = {}
    for label, bval in BETA_MODELS:
        t5[label] = phu.predictive_stability_table(all_fits[label], X_star)
        phu._log(f"  {label:<20s}  mean_d_1/2={t5[label]:.4f}", LOG_FILE)

    return {'table4': t4, 'table5': t5}


tables45 = phu.load_or_compute_json('_checkpoint_tables45.json', _compute_tables45, LOG_FILE, label='tables45')
table4 = tables45['table4']
table5 = tables45['table5']

[tables45] Loaded cached result from _checkpoint_tables45.json (skipping computation).


In [8]:
# (Table 4 now computed together with Table 5 in the cell above, since both share the same 5-model block fits.)

In [9]:
def _compute_table6():
    phu._log('\n=== STAGE: fitting LOBO-CV folds for Table 6 (5 models x 8 folds) ===', LOG_FILE)
    t6 = {}
    for label, bval in BETA_MODELS:
        model_this = model_std_block if bval is None else phu.make_model(beta=float(bval), tau0=tau0_block, k_trunc=250)
        safe_label = label.replace('*', 'star').replace('=', '_').replace(' ', '_')
        t6[label] = phu.lobo_cv_predictive_metrics(
            model_this, blocks, num_warmup=NUM_WARMUP, num_samples=NUM_SAMPLES,
            base_seed_mcmc=7000, target_accept_prob=TARGET_ACCEPT, max_tree_depth=MAX_TREE_DEPTH,
            n_pred_draws=200, log_file=LOG_FILE, label=label,
            checkpoint_prefix=f'_posteriors_table6_{safe_label}')
        m = t6[label]
        phu._log(f"  {label:<20s}  MLPD={m['mlpd']:.4f}  RMSE={m['rmse']:.4f}  "
                 f"Cov90={m['cov90']:.3f}  Cov95={m['cov95']:.3f}  Cov99={m['cov99']:.3f}  CRPS={m['crps']:.4f}", LOG_FILE)
    return t6

phu._log('\n=== Table 6: leave-one-block-out predictive accuracy ===', LOG_FILE)
table6 = phu.load_or_compute_json('_checkpoint_table6.json', _compute_table6, LOG_FILE, label='table6')


=== Table 6: leave-one-block-out predictive accuracy ===



=== STAGE: fitting LOBO-CV folds for Table 6 (5 models x 8 folds) ===


[Standard Bayes fold 1/8] Loaded cached posteriors from _posteriors_table6_Standard_Bayes_fold0.pkl (skipping computation).


  [Standard Bayes] LOBO fold 1/8 done


[Standard Bayes fold 2/8] Loaded cached posteriors from _posteriors_table6_Standard_Bayes_fold1.pkl (skipping computation).


...

  [beta=1.05] LOBO fold 4/8 done


[beta=1.05 fold 5/8] Saved posteriors to _posteriors_table6_beta_1.05_fold4.pkl.


  [beta=1.05] LOBO fold 5/8 done


[beta=1.05 fold 6/8] Saved posteriors to _posteriors_table6_beta_1.05_fold5.pkl.


  [beta=1.05] LOBO fold 6/8 done


In [ ]:
phu._log('\n=== Table 6: leave-one-block-out predictive accuracy ===', LOG_FILE)
table6 = {}
for label, bval in BETA_MODELS:
    model_this = model_std_block if bval is None else phu.make_model(beta=float(bval), tau0=tau0_block, k_trunc=250)
    table6[label] = phu.lobo_cv_predictive_metrics(
        model_this, blocks, num_warmup=NUM_WARMUP, num_samples=NUM_SAMPLES,
        base_seed_mcmc=7000, target_accept_prob=TARGET_ACCEPT, max_tree_depth=MAX_TREE_DEPTH,
        n_pred_draws=200, log_file=LOG_FILE, label=label)
    m = table6[label]
    phu._log(f"  {label:<20s}  MLPD={m['mlpd']:.4f}  RMSE={m['rmse']:.4f}  "
             f"Cov90={m['cov90']:.3f}  Cov95={m['cov95']:.3f}  Cov99={m['cov99']:.3f}  CRPS={m['crps']:.4f}", LOG_FILE)

## Final summary (Tables 4-6)

In [ ]:
lines = []
lines.append('=' * 100)
lines.append(f"{'Model':<20s}  {'L2':>8s}  {'Frob':>8s}  {'d_1/2(pred)':>12s}  {'MLPD':>8s}  {'RMSE':>7s}  {'Cov90':>6s}  {'Cov95':>6s}  {'Cov99':>6s}  {'CRPS':>7s}"
lines.append('-' * 100)
for label, bval in BETA_MODELS:
    t4, t5, t6 = table4[label], table5[label], table6[label]
    lines.append(f"{label:<20s}  {t4['mean_l2']:8.4f}  {t4['mean_frobenius']:8.4f}  {t5:12.4f}  "
                 f"{t6['mlpd']:8.4f}  {t6['rmse']:7.4f}  {t6['cov90']:6.3f}  {t6['cov95']:6.3f}  {t6['cov99']:6.3f}  {t6['crps']:7.4f}")
lines.append('=' * 100)
for line in lines:
    phu._log(line, LOG_FILE)